In [1]:
#Project 2 Lede Program
#Marcela Vilar Mota Santos

In [113]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
from opensky_api import OpenSkyApi, TokenManager
import geocoder
from datetime import datetime, timezone
import json

## getting the data from the world cup matches and basic info about the world cup

In [3]:
url = "https://www.zoho.com/toolkit/fifa-world-cup-2026.html?zredirect=f&zsrc=langdropdown&lb=pt-br&lb=pt-br"

In [4]:
table = pd.read_html(url)

In [119]:
stadiums = table[0]
stadiums.head(3)

,Host Country,City,Venue,Net Capacity
0,United States,New York/New Jersey,New York New Jersey Stadium,78576
1,United States,Kansas City,Kansas City Stadium,67513
2,United States,San Francisco Bay Area,San Francisco Bay Area Stadium,69391


In [122]:
stadiums.shape

(16, 4)

In [120]:
stadiums.sort_values(by="Net Capacity", ascending=False)
#New York New Jersey Stadium biggest stadium

,Host Country,City,Venue,Net Capacity
0,United States,New York/New Jersey,New York New Jersey Stadium,78576
13,Mexico,Mexico City,Mexico City Stadium,72766
6,United States,Dallas,Dallas Stadium,70122
7,United States,Los Angeles,Los Angeles Stadium,69650
2,United States,San Francisco Bay Area,San Francisco Bay Area Stadium,69391
5,United States,Houston,Houston Stadium,68311
1,United States,Kansas City,Kansas City Stadium,67513
3,United States,Atlanta,Atlanta Stadium,67382
8,United States,Philadelphia,Philadelphia Stadium,65827
10,United States,Seattle,Seattle Stadium,65123


In [6]:
prize = table[4]
prize.head(3)

,Final Position,Prize Money (per Team)
0,Champions (Winner),$50 million
1,Runner-up,$33 million
2,Third place,$29 million


In [7]:
groups = table[5]
groups.head(3)

,Group,Participating Nations
0,Group A,"Mexico, South Africa, South Korea, Czechia"
1,Group B,"Canada, Bosnia and Herzegovina, Switzerland, Q..."
2,Group C,"Brazil, Morocco, Scotland, Haiti"


In [8]:
matches = table[6][1:]
matches.head(3)

,Match,Date,Day,Group Stage,Team 1,vs,Team 2,Venue,City,Region
1,1,Jun 11,Thu,A,Mexico,vs,South Africa,Mexico City Stadium,Mexico City,Central
2,2,Jun 11,Thu,A,South Korea,vs,Czech Republic,Estadio Guadalajara,Guadalajara,Central
3,3,Jun 12,Fri,B,Canada,vs,Bosnia and Herzegovina,Toronto Stadium,Toronto,Eastern


In [121]:
matches.tail(5)

,Match,Date,Team 1,Team 2,Venue,City,phase
106,102,2026-07-15,Winner Match 99,Winner Match 100,Atlanta Stadium,Atlanta,Semifinals
107,"THIRD-PLACE MATCH (July 18, 2026)",NaT,"THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)",None
108,103,2026-07-18,Loser Match 101,Loser Match 102,Miami Stadium,Miami,Third Place
109,"FINAL (July 19, 2026)",NaT,"FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)",None
110,104,2026-07-19,Winner Match 101,Winner Match 102,New York New Jersey Stadium,New York/NJ,Final


In [9]:
#droping columns
matches = matches.drop(columns =['Day','Group Stage', 'vs', "Region"])

In [10]:
matches.head(3)

,Match,Date,Team 1,Team 2,Venue,City
1,1,Jun 11,Mexico,South Africa,Mexico City Stadium,Mexico City
2,2,Jun 11,South Korea,Czech Republic,Estadio Guadalajara,Guadalajara
3,3,Jun 12,Canada,Bosnia and Herzegovina,Toronto Stadium,Toronto


In [11]:
len(matches)

110

In [12]:
#finding irregular columns
matches["Match"][109]

'FINAL (July 19, 2026)'

In [13]:
matches["Match"][107]

'THIRD-PLACE MATCH (July 18, 2026)'

In [14]:
matches["Match"][104]

'SEMI-FINALS (July 14 – July 15, 2026)'

In [15]:
matches["Match"][99]

'QUARTER-FINALS (July 9 – July 11, 2026)'

In [16]:
matches["Match"][90]

'ROUND OF 16 (July 4 – July 7, 2026)'

In [17]:
matches["Match"][73]

'ROUND OF 32 (June 28 – July 3, 2026)'

In [18]:
#converting to datetime format
#adding the year and converting to datetime
matches["Date"] = pd.to_datetime(matches["Date"], format="%b %d", errors="coerce")

#formating as MM-DD-YYYY
matches["Date"] = matches["Date"].dt.strftime("%m-%d-2026")

In [19]:
matches.head(3)

,Match,Date,Team 1,Team 2,Venue,City
1,1,06-11-2026,Mexico,South Africa,Mexico City Stadium,Mexico City
2,2,06-11-2026,South Korea,Czech Republic,Estadio Guadalajara,Guadalajara
3,3,06-12-2026,Canada,Bosnia and Herzegovina,Toronto Stadium,Toronto


## separating by phase 
##### project due date is july 12, so i can only analyse until the end of the round of 16 (july 7) and maybe quarter finals (july 11)

In [20]:
#creating a new empty column
matches["phase"] = None

In [21]:
matches.head(3)

,Match,Date,Team 1,Team 2,Venue,City,phase
1,1,06-11-2026,Mexico,South Africa,Mexico City Stadium,Mexico City,None
2,2,06-11-2026,South Korea,Czech Republic,Estadio Guadalajara,Guadalajara,None
3,3,06-12-2026,Canada,Bosnia and Herzegovina,Toronto Stadium,Toronto,None


In [22]:
matches["Date"].dtype

<StringDtype(storage='python', na_value=nan)>

In [23]:
#transforming to datetime element so the for loop works
matches["Date"] = pd.to_datetime(matches['Date'])

In [24]:
matches["Date"].dtype

dtype('<M8[us]')

In [25]:
#filling the new column
for index, row in matches.iterrows(): 
    if pd.Timestamp("2026-06-11") <= row["Date"] <= pd.Timestamp("2026-06-27"):
       matches.at[index, "phase"] = "Group Stage"
    elif pd.Timestamp("2026-06-28") <= row["Date"] <= pd.Timestamp("2026-07-03"):
       matches.at[index, "phase"] = "Round of 32"
    elif pd.Timestamp("2026-07-04") <= row["Date"] <= pd.Timestamp("2026-07-07"):
       matches.at[index, "phase"] = "Round of 16"
    elif pd.Timestamp("2026-07-09") <= row["Date"] <= pd.Timestamp("2026-07-11"):
       matches.at[index, "phase"] = "Quarterfinals"
    elif pd.Timestamp("2026-07-14") <= row["Date"] <= pd.Timestamp("2026-07-15"):
       matches.at[index, "phase"] = "Semifinals"
    elif pd.Timestamp("2026-07-18") == row["Date"]:
       matches.at[index, "phase"] = "Third Place"
    elif pd.Timestamp("2026-07-19") == row["Date"]:
       matches.at[index, "phase"] = "Final"

In [26]:
matches["phase"].unique()

array(['Group Stage', None, 'Round of 32', 'Round of 16', 'Quarterfinals',
       'Semifinals', 'Third Place', 'Final'], dtype=object)

In [27]:
matches[matches["phase"].isna()]

,Match,Date,Team 1,Team 2,Venue,City,phase
73,"ROUND OF 32 (June 28 – July 3, 2026)",NaT,"ROUND OF 32 (June 28 – July 3, 2026)","ROUND OF 32 (June 28 – July 3, 2026)","ROUND OF 32 (June 28 – July 3, 2026)","ROUND OF 32 (June 28 – July 3, 2026)",None
90,"ROUND OF 16 (July 4 – July 7, 2026)",NaT,"ROUND OF 16 (July 4 – July 7, 2026)","ROUND OF 16 (July 4 – July 7, 2026)","ROUND OF 16 (July 4 – July 7, 2026)","ROUND OF 16 (July 4 – July 7, 2026)",None
99,"QUARTER-FINALS (July 9 – July 11, 2026)",NaT,"QUARTER-FINALS (July 9 – July 11, 2026)","QUARTER-FINALS (July 9 – July 11, 2026)","QUARTER-FINALS (July 9 – July 11, 2026)","QUARTER-FINALS (July 9 – July 11, 2026)",None
104,"SEMI-FINALS (July 14 – July 15, 2026)",NaT,"SEMI-FINALS (July 14 – July 15, 2026)","SEMI-FINALS (July 14 – July 15, 2026)","SEMI-FINALS (July 14 – July 15, 2026)","SEMI-FINALS (July 14 – July 15, 2026)",None
107,"THIRD-PLACE MATCH (July 18, 2026)",NaT,"THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)",None
109,"FINAL (July 19, 2026)",NaT,"FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)",None


In [28]:
matches["phase"].value_counts() #checking if it worked

phase
Group Stage      72
Round of 32      16
Round of 16       8
Quarterfinals     4
Semifinals        2
Third Place       1
Final             1
Name: count, dtype: int64

In [29]:
matches.to_csv("matches.csv", index=False)

## hiding my geocoder API key 

In [30]:
#hiding my api key
# create a file called `.env` (# it's a hidden file)
# you also will want to create a file called .gitignore (it is also hidden)
# and put this in it (without the #)
# .env 
# that's to tell Git not to put your .env file (with your API key) onto Github

In [31]:
load_dotenv()

True

In [32]:
ls

 O volume na unidade C ‚ OS
 O N£mero de S‚rie do Volume ‚ 44F5-4363

 Pasta de C:\Users\samue\OneDrive\Desktop\Lede\Projects\world cup

07/09/2026  07:41 AM    <DIR>          .
07/06/2026  02:54 PM    <DIR>          ..
07/08/2026  06:58 PM               116 .env
07/05/2026  11:58 PM             4,706 .gitignore
07/08/2026  05:00 PM    <DIR>          .ipynb_checkpoints
07/05/2026  11:49 PM                60 api key geocoder.txt
07/08/2026  07:27 PM                83 credentials.json
07/08/2026  03:59 PM             7,611 df_geo.csv
07/03/2026  05:04 PM             4,727 extract_worldcup.py
07/03/2026  03:59 PM            44,443 fifa-world-cup-2026.ics
07/08/2026  05:00 PM             3,127 first_matches.csv
07/09/2026  07:41 AM             9,399 matches.csv
07/03/2026  06:23 PM               137 mentorship july 3.txt
07/05/2026  11:53 PM                60 untitled.txt
07/09/2026  07:41 AM           542,214 worldcup.ipynb
07/03/2026  05:09 PM            28,228 worldcup_PDF.ipynb
07/08/2

In [33]:
api_key_geocoder = os.environ["api_key_geocoder"]

In [34]:
api_key_geocoder[:9]+"..."

'AIzaSyDI7...'

In [35]:
#making a new df with only the venue and city, because it may be easier for the geocoder to read 
df_geo = matches[['Venue', 'City']]
df_geo.head(3)

,Venue,City
1,Mexico City Stadium,Mexico City
2,Estadio Guadalajara,Guadalajara
3,Toronto Stadium,Toronto


In [36]:
#getting the country so geocoder would work
stadiums.head(3)

,Host Country,Host City,Official FIFA Stadium Designation,Net Capacity
0,United States,New York/New Jersey,New York New Jersey Stadium,78576
1,United States,Kansas City,Kansas City Stadium,67513
2,United States,San Francisco Bay Area,San Francisco Bay Area Stadium,69391


In [37]:
#changing the name of the column so it could merge
stadiums.rename(columns={"Host City":"City"}, inplace=True)

In [38]:
#changing the name of the column so it could merge
stadiums.rename(columns={"Official FIFA Stadium Designation":"Venue"}, inplace=True)

In [39]:
stadiums.head(3)

,Host Country,City,Venue,Net Capacity
0,United States,New York/New Jersey,New York New Jersey Stadium,78576
1,United States,Kansas City,Kansas City Stadium,67513
2,United States,San Francisco Bay Area,San Francisco Bay Area Stadium,69391


In [40]:
df_geo = df_geo.merge(
    stadiums[["City", "Host Country"]],
    on="City",
    how="left"
)

In [41]:
df_geo.head(3)

,Venue,City,Host Country
0,Mexico City Stadium,Mexico City,Mexico
1,Estadio Guadalajara,Guadalajara,Mexico
2,Toronto Stadium,Toronto,Canada


## geocoding

In [43]:
df_geo[df_geo["Venue"] == "New York New Jersey Stadium"]
#it didn't put the country in those rows for some reason

,Venue,City,Host Country
5,New York New Jersey Stadium,New York/NJ,NaN
16,New York New Jersey Stadium,New York/NJ,NaN
40,New York New Jersey Stadium,New York/NJ,NaN
55,New York New Jersey Stadium,New York/NJ,NaN
66,New York New Jersey Stadium,New York/NJ,NaN
77,New York New Jersey Stadium,New York/NJ,NaN
92,New York New Jersey Stadium,New York/NJ,NaN
109,New York New Jersey Stadium,New York/NJ,NaN


In [44]:
df_geo.loc[df_geo["Venue"] == "New York New Jersey Stadium",
    "Host Country"
] = "United States"
#.loc is the standard pandas way to select rows and columns by their labels and modify values
#.loc uses that Boolean Series to identify which rows to update

In [45]:
df_geo[df_geo["Venue"] == "New York New Jersey Stadium"]
#checking if it worked

,Venue,City,Host Country
5,New York New Jersey Stadium,New York/NJ,United States
16,New York New Jersey Stadium,New York/NJ,United States
40,New York New Jersey Stadium,New York/NJ,United States
55,New York New Jersey Stadium,New York/NJ,United States
66,New York New Jersey Stadium,New York/NJ,United States
77,New York New Jersey Stadium,New York/NJ,United States
92,New York New Jersey Stadium,New York/NJ,United States
109,New York New Jersey Stadium,New York/NJ,United States


In [46]:
df_geo.to_csv("df_geo.csv", index=False)

In [47]:
#getting the latitude and longitude
for index, row in df_geo.iterrows():
    full_address = f"{row['Venue']}, {row['City']}, {row['Host Country']}"
    print(full_address)
    
    geo = geocoder.google(full_address, key=api_key_geocoder)
    print("Status:", geo.status)
    print("OK:", geo.ok)
    print("LatLng:", geo.latlng)
    
    if geo.ok:
        df_geo.loc[index, "lat"] = geo.latlng[0]
        df_geo.loc[index, "long"] = geo.latlng[1]
    else:
        print(f"Failed for {full_address}: {geo.status}")

df_geo

Mexico City Stadium, Mexico City, Mexico
Status: OK
OK: True
LatLng: [19.3028607, -99.1505277]
Estadio Guadalajara, Guadalajara, Mexico
Status: OK
OK: True
LatLng: [20.6817764, -103.4626455]
Toronto Stadium, Toronto, Canada
Status: OK
OK: True
LatLng: [43.6331948, -79.41856349999999]
Los Angeles Stadium, Los Angeles, United States
Status: OK
OK: True
LatLng: [33.9534765, -118.3390235]
San Francisco Bay Area Stadium, San Francisco Bay Area, United States
Status: OK
OK: True
LatLng: [37.4033165, -121.9693774]
New York New Jersey Stadium, New York/NJ, United States
Status: OK
OK: True
LatLng: [40.8135064, -74.074457]
Boston Stadium, Boston, United States
Status: OK
OK: True
LatLng: [42.09092740000001, -71.264363]
BC Place Vancouver, Vancouver, Canada
Status: OK
OK: True
LatLng: [49.2827291, -123.1207375]
Houston Stadium, Houston, United States
Status: OK
OK: True
LatLng: [29.6847219, -95.41070739999999]
Dallas Stadium, Dallas, United States
Status: OK
OK: True
LatLng: [32.7480281, -97.092

,Venue,City,Host Country,lat,long
0,Mexico City Stadium,Mexico City,Mexico,19.302861,-99.150528
1,Estadio Guadalajara,Guadalajara,Mexico,20.681776,-103.462645
2,Toronto Stadium,Toronto,Canada,43.633195,-79.418563
3,Los Angeles Stadium,Los Angeles,United States,33.953477,-118.339023
4,San Francisco Bay Area Stadium,San Francisco Bay Area,United States,37.403317,-121.969377
...,...,...,...,...,...
105,Atlanta Stadium,Atlanta,United States,33.755323,-84.400590
106,"THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)",NaN,18.684809,100.800005
107,Miami Stadium,Miami,United States,25.761680,-80.191790
108,"FINAL (July 19, 2026)","FINAL (July 19, 2026)",NaN,NaN,NaN


In [48]:
df_geo.to_csv("df_geo.csv", index=False)

## doing some analysis

In [49]:
matches.head()

,Match,Date,Team 1,Team 2,Venue,City,phase
1,1,2026-06-11,Mexico,South Africa,Mexico City Stadium,Mexico City,Group Stage
2,2,2026-06-11,South Korea,Czech Republic,Estadio Guadalajara,Guadalajara,Group Stage
3,3,2026-06-12,Canada,Bosnia and Herzegovina,Toronto Stadium,Toronto,Group Stage
4,4,2026-06-12,USA,Paraguay,Los Angeles Stadium,Los Angeles,Group Stage
5,5,2026-06-13,Qatar,Switzerland,San Francisco Bay Area Stadium,San Francisco Bay Area,Group Stage


In [50]:
matches_by_city = matches["City"].value_counts()
matches_by_city
#Dallas hosted more games

City
Dallas                                     9
Los Angeles                                8
New York/NJ                                8
Atlanta                                    8
Miami                                      8
Boston                                     7
Vancouver                                  7
Mexico City                                6
San Francisco Bay Area                     6
Houston                                    6
Philadelphia                               6
Seattle                                    6
Kansas City                                6
Toronto                                    5
Guadalajara                                4
Monterrey                                  4
ROUND OF 32 (June 28 – July 3, 2026)       1
ROUND OF 16 (July 4 – July 7, 2026)        1
QUARTER-FINALS (July 9 – July 11, 2026)    1
SEMI-FINALS (July 14 – July 15, 2026)      1
THIRD-PLACE MATCH (July 18, 2026)          1
FINAL (July 19, 2026)                      1
Name:

In [51]:
stadiums = matches["Venue"].value_counts()
stadiums

Venue
Dallas Stadium                             9
Los Angeles Stadium                        8
New York New Jersey Stadium                8
Atlanta Stadium                            8
Miami Stadium                              8
Boston Stadium                             7
BC Place Vancouver                         7
Mexico City Stadium                        6
San Francisco Bay Area Stadium             6
Houston Stadium                            6
Philadelphia Stadium                       6
Seattle Stadium                            6
Kansas City Stadium                        6
Toronto Stadium                            5
Estadio Guadalajara                        4
Estadio Monterrey                          4
ROUND OF 32 (June 28 – July 3, 2026)       1
ROUND OF 16 (July 4 – July 7, 2026)        1
QUARTER-FINALS (July 9 – July 11, 2026)    1
SEMI-FINALS (July 14 – July 15, 2026)      1
THIRD-PLACE MATCH (July 18, 2026)          1
FINAL (July 19, 2026)                      1
Name

In [52]:
#I decided that i'm working with the first week of the world cup
group_stage = matches[matches["phase"] == "Group Stage"]
group_stage

,Match,Date,Team 1,Team 2,Venue,City,phase
1,1,2026-06-11,Mexico,South Africa,Mexico City Stadium,Mexico City,Group Stage
2,2,2026-06-11,South Korea,Czech Republic,Estadio Guadalajara,Guadalajara,Group Stage
3,3,2026-06-12,Canada,Bosnia and Herzegovina,Toronto Stadium,Toronto,Group Stage
4,4,2026-06-12,USA,Paraguay,Los Angeles Stadium,Los Angeles,Group Stage
5,5,2026-06-13,Qatar,Switzerland,San Francisco Bay Area Stadium,San Francisco Bay Area,Group Stage
...,...,...,...,...,...,...,...
68,68,2026-06-27,Croatia,Ghana,Philadelphia Stadium,Philadelphia,Group Stage
69,69,2026-06-27,Jordan,Argentina,Dallas Stadium,Dallas,Group Stage
70,70,2026-06-27,Algeria,Austria,Kansas City Stadium,Kansas City,Group Stage
71,71,2026-06-27,Colombia,Portugal,Miami Stadium,Miami,Group Stage


In [53]:
group_stage.drop(columns="phase")

,Match,Date,Team 1,Team 2,Venue,City
1,1,2026-06-11,Mexico,South Africa,Mexico City Stadium,Mexico City
2,2,2026-06-11,South Korea,Czech Republic,Estadio Guadalajara,Guadalajara
3,3,2026-06-12,Canada,Bosnia and Herzegovina,Toronto Stadium,Toronto
4,4,2026-06-12,USA,Paraguay,Los Angeles Stadium,Los Angeles
5,5,2026-06-13,Qatar,Switzerland,San Francisco Bay Area Stadium,San Francisco Bay Area
...,...,...,...,...,...,...
68,68,2026-06-27,Croatia,Ghana,Philadelphia Stadium,Philadelphia
69,69,2026-06-27,Jordan,Argentina,Dallas Stadium,Dallas
70,70,2026-06-27,Algeria,Austria,Kansas City Stadium,Kansas City
71,71,2026-06-27,Colombia,Portugal,Miami Stadium,Miami


In [54]:
matches.groupby("Date")["Match"].count().sort_values(ascending=False)
#number of games per day during the whole world cup

Date
2026-06-26    6
2026-06-25    6
2026-06-27    6
2026-06-24    6
2026-06-20    4
2026-06-19    4
2026-06-13    4
2026-06-14    4
2026-06-15    4
2026-06-16    4
2026-06-23    4
2026-06-22    4
2026-06-18    4
2026-06-17    4
2026-06-21    4
2026-06-30    3
2026-07-03    3
2026-06-29    3
2026-07-01    3
2026-07-02    3
2026-06-12    2
2026-06-11    2
2026-07-07    2
2026-07-04    2
2026-07-05    2
2026-07-06    2
2026-07-11    2
2026-06-28    1
2026-07-09    1
2026-07-10    1
2026-07-14    1
2026-07-15    1
2026-07-18    1
2026-07-19    1
Name: Match, dtype: int64

In [55]:
group_stage.groupby("Date")["Match"].count().sort_values(ascending=False)
#number of games per day during the group phase

Date
2026-06-24    6
2026-06-26    6
2026-06-25    6
2026-06-27    6
2026-06-23    4
2026-06-13    4
2026-06-14    4
2026-06-15    4
2026-06-16    4
2026-06-21    4
2026-06-19    4
2026-06-17    4
2026-06-18    4
2026-06-20    4
2026-06-22    4
2026-06-11    2
2026-06-12    2
Name: Match, dtype: int64

## getting data from the Air traffic (OPenSky)

In [56]:
group_stage.groupby("Date")['Match'].count()

Date
2026-06-11    2
2026-06-12    2
2026-06-13    4
2026-06-14    4
2026-06-15    4
2026-06-16    4
2026-06-17    4
2026-06-18    4
2026-06-19    4
2026-06-20    4
2026-06-21    4
2026-06-22    4
2026-06-23    4
2026-06-24    6
2026-06-25    6
2026-06-26    6
2026-06-27    6
Name: Match, dtype: int64

In [57]:
#first day of the world cup 
group_stage[["Date", "City"]].head(2)

,Date,City
1,2026-06-11,Mexico City
2,2026-06-11,Guadalajara


In [58]:
airports_mexico_city = {"airport_1" : "Benito Juárez International Airport (MEX)",
                        "ICAO_code_1" : "MMMX",
                        "airport_2" : "Felipe Ángeles International Airport (AIFA)",
                        "ICAO_code_2" : "MMSM"}

In [59]:
airports_mexico_city.keys()

dict_keys(['airport_1', 'ICAO_code_1', 'airport_2', 'ICAO_code_2'])

In [60]:
airports_guadalajara = {"airport": "Guadalajara International Airport",
                        "ICAO_code": "MMGL"}

In [86]:
#begin and end are not dates or ISO timestamps
#must be Unix timestamps (seconds since January 1, 1970 UTC)

#From June 10 at 00h to June 11 at 10am in NY
#first game started at 15h June 11

begin = int(datetime(2026, 6, 10, 0, 0, tzinfo=timezone.utc).timestamp())
end = int(datetime(2026, 6, 10, 1, 0, tzinfo=timezone.utc).timestamp())

print(begin)
print(end)

1781049600
1781053200


In [87]:
with OpenSkyApi(token_manager=TokenManager.from_json_file("credentials.json")) as api:
    states = api.get_states()

In [89]:
api = OpenSkyApi()
arrivals = api.get_arrivals_by_airport("KJFK", begin, end)
print(arrivals)
#didn't work with the library

None


In [79]:
#example from the website
api = OpenSkyApi()
arrivals = api.get_arrivals_by_airport("EDDF", 1517227200, 1517230800)
print("Arrivals:")
for flight in arrivals:
    print(flight)

Arrivals:


TypeError: 'NoneType' object is not iterable

In [81]:
print(arrivals)
print(type(arrivals))

None
<class 'NoneType'>


In [82]:
if arrivals:
    for flight in arrivals:
        print(flight.callsign)
else:
    print("No flights returned")

No flights returned


In [74]:
#test2
api = OpenSkyApi()
states = api.get_states()
for s in states.states:
    print("(%r, %r, %r, %r)" % (s.longitude, s.latitude, s.baro_altitude, s.velocity))

(12.4039, 37.9645, 11582.4, 227.93)
(-7.2917, 39.1197, 11277.6, 227.42)
(-77.2215, 41.9609, 10957.56, 218.22)
(-81.6979, 27.6458, 762, 38.92)
(24.3144, 39.8427, 11590.02, 213.05)
(18.8428, 36.494, 11285.22, 245.13)
(3.6231, 46.9774, 10340.34, 234.3)
(49.0422, 26.7642, 12192, 257.23)
(20.8603, 52.5928, 419.1, 40.04)
(-82.7917, 34.5309, 4213.86, 168.41)
(2.3618, 48.4698, 1493.52, 147.33)
(14.5324, 54.642, 2903.22, 63.28)
(76.7408, 26.8158, 6012.18, 199.09)
(106.4843, -6.0837, 1272.54, 107.28)
(7.0117, 50.7101, 11894.82, 233.89)
(-0.9844, 43.8617, 11887.2, 222.67)
(131.3211, 33.4127, 6400.8, 228.96)
(-122.3647, 47.6035, 381, 61.77)
(-78.4915, 34.4671, 2987.04, 159.53)
(77.264, 12.1938, 8229.6, 190.55)
(-77.0363, 38.7886, 708.66, 108.03)
(28.1522, -26.0225, 3268.98, 129.76)
(-105.1202, 40.0494, 2225.04, 66.27)
(-75.2342, 39.8778, None, 5.66)
(-129.5408, 43.669, 11277.6, 235.94)
(-80.9911, 38.5384, 11582.4, 215.78)
(76.8411, 27.6506, 6103.62, 204.84)
(-73.4035, 45.5182, None, 0)
(76.9919, 1

In [ ]:
#unix_time(-date)

In [ ]:
# def unix_time(date):
#     year = date.dt.year
#     month = month.dt.month
#     day = day.dt.day
#     begin = int(datetime(year, month, day, 0, 0, tzinfo=timezone.utc).timestamp())
#     end = int(datetime(year, month, day, 10, 0, tzinfo=timezone.utc).timestamp())
#     print(begin)
#     print(end)

In [114]:
with open("credentials.json") as f:
    creds = json.load(f)

CLIENT_ID = creds["client_id"]
CLIENT_SECRET = creds["client_secret"]

TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"
API_URL = "https://opensky-network.org/api/flights/arrival"

def get_token():
    r = requests.post(
        TOKEN_URL,
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
        },
    )
    r.raise_for_status()
    return r.json()["access_token"]

token = get_token()

In [112]:
def get_flights(airport, begin, end):
    token = get_token()
    headers = {"Authorization": f"Bearer {token}"}
    resp = requests.get(
        API_URL,
        params={"airport": airport, "begin": begin, "end": end},
        headers=headers,
    )
 
    if resp.status_code == 404:
        print("No flights found for this window.")
        return []
 
    resp.raise_for_status()
    return resp.json()
 
 
if __name__ == "__main__":
    airport = "MMMX"  # Mexico City International Airport
    begin = 1781049600
    end = 1781053200
 
    flights = get_flights(airport, begin, end)
    print(f"{len(flights)} flights found for {airport}")
 
    for f in flights:
        print(f)

24 flights found for MMMX
{'icao24': 'aa9bb7', 'firstSeen': 1781011954, 'estDepartureAirport': 'LEBL', 'lastSeen': 1781053045, 'estArrivalAirport': 'MMMX', 'callsign': 'AMX038  ', 'estDepartureAirportHorizDistance': 1482, 'estDepartureAirportVertDistance': 34, 'estArrivalAirportHorizDistance': 1858, 'estArrivalAirportVertDistance': 58, 'departureAirportCandidatesCount': 0, 'arrivalAirportCandidatesCount': 1}
{'icao24': '0d0f91', 'firstSeen': 1781051529, 'estDepartureAirport': None, 'lastSeen': 1781052892, 'estArrivalAirport': 'MMMX', 'callsign': 'VIV1169 ', 'estDepartureAirportHorizDistance': None, 'estDepartureAirportVertDistance': None, 'estArrivalAirportHorizDistance': 345, 'estArrivalAirportVertDistance': 96, 'departureAirportCandidatesCount': 0, 'arrivalAirportCandidatesCount': 1}
{'icao24': 'a1245e', 'firstSeen': 1781039389, 'estDepartureAirport': 'KIAD', 'lastSeen': 1781052714, 'estArrivalAirport': 'MMMX', 'callsign': 'AMX456  ', 'estDepartureAirportHorizDistance': 1609, 'estDep

In [92]:
#hiding my airlab api key 
load_dotenv()

True

In [93]:
api_key_airlab = os.environ["api_key_airlab"]

In [94]:
api_key_airlab[:9]+"..."

'39f9f1b1-...'

## analysing ITA database https://www.trade.gov/us-international-air-travel-statistics-i-92-data

In [97]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [102]:
ita = pd.read_excel("ITA.xlsx", header=3)

In [103]:
ita.head()

,Regions,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,YTD,YTD (1),Share
0,Europe,971828,936337,1636773,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3544938,0.044818,0.1444
1,Caribbean,851394,924719,1178575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2954688,-0.020855,0.120356
2,Asia,597825,577694,805164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1980683,0.102466,0.080681
3,South America,261135,273955,313664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,848754,0.00102,0.034573
4,Central America,467792,465191,579988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1512971,0.103998,0.061629


In [106]:
ita_2 = pd.read_excel("ITA_monthly.xlsx")

In [107]:
ita_2.head()

,U.S. CITIZEN TRAVEL TO INTERNATIONAL REGIONS,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,"Released: June 17, 2026",NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,YTD
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Total,% Change,Market
2,Regions,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,YTD,YTD (1),Share
3,Europe,971828,936337,1636773,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3544938,0.044818,0.1444
4,Caribbean,851394,924719,1178575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2954688,-0.020855,0.120356
